# 2단계: 대중교통 공백 분석

**목적:** 지하철 막차 이후 대중교통이 끊기는 구간과 오피스 밀집 구역을 교차해
자율주행 택시 수요가 발생하는 '교통 공백 + 오피스' 구역을 도출

**활용 데이터:**
- 서울시 지하철 역별 시간대별 승하차 인원 (OA-12921) — 열린데이터광장
- 심야버스(올빼미버스) 노선 현황 — 열린데이터광장
- 1단계 산출물: stage01_office_index.csv

**산출물:** 대중교통 공백 지수 + 공백 구역 지도

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
from sklearn.preprocessing import MinMaxScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os, warnings
warnings.filterwarnings('ignore')

# 한글 폰트 (Linux에서는 NanumGothic 사용)
import subprocess
try:
    subprocess.run(['fc-list', ':lang=ko'], capture_output=True)
    plt.rcParams['font.family'] = ['NanumGothic', 'DejaVu Sans']
except:
    pass
plt.rcParams['axes.unicode_minus'] = False

DATA_DIR   = 'data/'
OUTPUT_DIR = 'output/'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('라이브러리 로드 완료')


라이브러리 로드 완료


## 1. 데이터 로드

In [2]:
# ── 1단계 결과 불러오기
df_office = pd.read_csv(OUTPUT_DIR + 'stage01_office_index.csv', dtype={'행정동코드': str})
df_office['행정동코드'] = df_office['행정동코드'].str.zfill(8)
print('1단계 결과:', df_office.shape, '/ 컬럼:', list(df_office.columns))

# ── 지하철 시간대별 승하차 인원
# 컬럼: 사용월, 호선명, 지하철역, 04시-05시 승차인원 ... 03시-04시 하차인원
df_subway = pd.read_csv(
    DATA_DIR + '서울시 지하철 호선별 역별 시간대별 승하차 인원 정보.csv',
    encoding='cp949'
)
print('지하철 데이터:', df_subway.shape)


1단계 결과: (426, 6) / 컬럼: ['행정동코드', '행정동명', '오피스_종사자수', '전체_종사자수', '오피스_비율', '오피스_밀집도_지수']


지하철 데이터: (81111, 52)


In [3]:
# ── 올빼미버스 노선 (N으로 시작하는 노선만 필터)
df_bus_route = pd.read_csv(DATA_DIR + '서울시 버스 노선 정보 조회.csv', encoding='cp949')
df_owl_route = df_bus_route[df_bus_route['노선명'].str.startswith('N', na=False)].copy()
print(f'올빼미버스 노선 수: {df_owl_route["노선명"].nunique()}개')

# ── 버스정류소 위치정보 (X좌표=경도, Y좌표=위도 — WGS84)
df_stop = pd.read_csv(DATA_DIR + '서울시 버스정류소 위치정보.csv', encoding='cp949')
# 올빼미버스 정류장만 추출 (노선 데이터의 NODE_ID로 조인)
owl_node_ids = set(df_owl_route['NODE_ID'].astype(str))
df_owl_stop = df_stop[df_stop['노드 ID'].astype(str).isin(owl_node_ids)].copy()

# 중복 제거 (같은 정류장에 여러 노선)
df_owl_stop = df_owl_stop.drop_duplicates(subset='노드 ID')
print(f'올빼미버스 정류장 수: {len(df_owl_stop)}개')


올빼미버스 노선 수: 18개
올빼미버스 정류장 수: 1494개


In [4]:
# ── 행정동 경계 Shapefile 로드
gdf_dong = gpd.read_file(DATA_DIR + 'bnd_dong_11_2025_2Q.shp')
if gdf_dong.crs is None or gdf_dong.crs.to_epsg() != 4326:
    gdf_dong = gdf_dong.to_crs('EPSG:4326')

gdf_dong = gdf_dong.rename(columns={'ADM_CD': '행정동코드', 'ADM_NM': '행정동명'})
gdf_dong['행정동코드'] = gdf_dong['행정동코드'].astype(str).str.zfill(8)
print(f'행정동 경계 로드 완료: {len(gdf_dong)}개 동')


행정동 경계 로드 완료: 426개 동


## 2. 지하철 막차 이후 승하차 급감 분석

In [5]:
# ── 지하철 막차 이후 심야 이용 패턴 분석
# 막차 직전(22~23시) vs 직후(23~02시) 비교

before_cols = ['22시-23시 승차인원', '23시-24시 승차인원']   # 막차 직전
after_cols  = ['00시-01시 승차인원', '01시-02시 승차인원']   # 막차 이후

# 존재하는 컬럼만 사용
before_cols = [c for c in before_cols if c in df_subway.columns]
after_cols  = [c for c in after_cols  if c in df_subway.columns]

df_subway['막차전_합'] = df_subway[before_cols].sum(axis=1)
df_subway['막차후_합'] = df_subway[after_cols].sum(axis=1)

# 역별 월평균 집계
df_station = df_subway.groupby('지하철역').agg(
    막차전_평균=('막차전_합', 'mean'),
    막차후_평균=('막차후_합', 'mean'),
).reset_index()

# 잔존율: 막차 후 수요 / 막차 전 수요 (낮을수록 심야 교통 공백 클 가능성)
df_station['막차후_잔존율'] = df_station['막차후_평균'] / (df_station['막차전_평균'] + 1)

print(f'분석 역 수: {len(df_station)}개')
print('잔존율 통계:')
print(df_station['막차후_잔존율'].describe().round(4))


분석 역 수: 600개
잔존율 통계:
count    600.0000
mean       0.0345
std        0.0183
min        0.0000
25%        0.0206
50%        0.0352
75%        0.0466
max        0.1311
Name: 막차후_잔존율, dtype: float64


In [6]:
# ── 시각화: 시간대별 전체 승차 추이 (전체 합산)
night_ride_cols = [c for c in df_subway.columns if '승차인원' in c]
hourly_sum = df_subway[night_ride_cols].sum()

# 컬럼명에서 시간대 추출
hour_labels = [c.split('시-')[0].split(' ')[-1] + 'h' for c in night_ride_cols]

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(range(len(night_ride_cols)), hourly_sum.values / 1e6, color='steelblue', alpha=0.8)

# 22시 이후 강조
for i, label in enumerate(hour_labels):
    h = int(label.replace('h',''))
    if h >= 22 or h <= 3:
        bars[i].set_color('tomato')

ax.set_xticks(range(len(night_ride_cols)))
ax.set_xticklabels(hour_labels, rotation=45, fontsize=8)
ax.set_xlabel('Time')
ax.set_ylabel('Ridership (million)')
ax.set_title('Seoul Subway Hourly Ridership (red=night hours)')
ax.axvline(x=[i for i, l in enumerate(hour_labels) if '22' in l][0] - 0.5,
           color='red', linestyle='--', alpha=0.5, label='22h cutoff')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR + '02_subway_hourly.png', dpi=150, bbox_inches='tight')
plt.close()
print('시간대별 승차 추이 저장 완료 → output/02_subway_hourly.png')


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


...

findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


시간대별 승차 추이 저장 완료 → output/02_subway_hourly.png


## 3. 올빼미버스 커버리지 분석 (버퍼 500m)

In [7]:
# ── 올빼미버스 정류장 GeoDataFrame 생성 (WGS84 → UTM-K 변환 후 500m 버퍼)
gdf_owl = gpd.GeoDataFrame(
    df_owl_stop,
    geometry=gpd.points_from_xy(df_owl_stop['X좌표'], df_owl_stop['Y좌표']),
    crs='EPSG:4326'
)

# 미터 단위 좌표계로 변환 (EPSG:5179 UTM-K)
gdf_owl_m  = gdf_owl.to_crs('EPSG:5179')
gdf_dong_m = gdf_dong.to_crs('EPSG:5179')

# 500m 버퍼
gdf_owl_buf = gdf_owl_m.copy()
gdf_owl_buf['geometry'] = gdf_owl_m.geometry.buffer(500)

# 버퍼 유니온 (전체 커버리지 영역)
owl_coverage = gdf_owl_buf.geometry.unary_union
print(f'올빼미버스 커버리지 면적: {owl_coverage.area / 1e6:.1f} km²')


올빼미버스 커버리지 면적: 263.3 km²


In [8]:
# ── 행정동 중심점 기준 커버리지 여부 계산
gdf_dong_m['centroid'] = gdf_dong_m.geometry.centroid
gdf_dong_m['owl_covered'] = gdf_dong_m['centroid'].within(owl_coverage)

# 면적 비율도 계산 (더 정밀)
def coverage_ratio(dong_geom, coverage):
    try:
        inter = dong_geom.intersection(coverage)
        return inter.area / dong_geom.area
    except:
        return 0.0

gdf_dong_m['owl_coverage_ratio'] = gdf_dong_m.geometry.apply(
    lambda g: coverage_ratio(g, owl_coverage)
)

covered_count = gdf_dong_m['owl_covered'].sum()
print(f'올빼미버스 커버 행정동: {covered_count}개 / {len(gdf_dong_m)}개')
print(f'미커버 행정동: {len(gdf_dong_m) - covered_count}개')
print(f'평균 커버리지 비율: {gdf_dong_m["owl_coverage_ratio"].mean():.1%}')


올빼미버스 커버 행정동: 249개 / 426개
미커버 행정동: 177개
평균 커버리지 비율: 56.2%


## 4. 공백 지수 산출 (오피스 밀집 × 교통 공백 교차)

In [9]:
# ── 대중교통 공백 지수 = 올빼미버스 미커버 정도
# owl_gap = 1 - coverage_ratio → 높을수록 교통 공백 심함

# WGS84로 복원
gdf_result = gdf_dong_m.to_crs('EPSG:4326')
gdf_result['owl_gap'] = 1 - gdf_result['owl_coverage_ratio']

# Min-Max 정규화 → 대중교통공백_지수 (0~1)
scaler = MinMaxScaler()
gdf_result['대중교통공백_지수'] = scaler.fit_transform(gdf_result[['owl_gap']])

# 1단계 결과와 병합
df_transport = gdf_result[['행정동코드', '행정동명', 'owl_covered', 'owl_coverage_ratio', '대중교통공백_지수']].copy()

print('대중교통 공백 지수 산출 완료')
print('상위 10개 (교통 공백 큰 지역):')
print(df_transport.nlargest(10, '대중교통공백_지수')[['행정동명', 'owl_coverage_ratio', '대중교통공백_지수']].to_string(index=False))


대중교통 공백 지수 산출 완료
상위 10개 (교통 공백 큰 지역):
 행정동명  owl_coverage_ratio  대중교통공백_지수
  부암동                 0.0        1.0
  평창동                 0.0        1.0
금호1가동                 0.0        1.0
금호4가동                 0.0        1.0
 수유2동                 0.0        1.0
 삼각산동                 0.0        1.0
  창3동                 0.0        1.0
 중계본동                 0.0        1.0
 중계1동                 0.0        1.0
 중계4동                 0.0        1.0


## 5. 지도 시각화 및 결과 저장

In [10]:
# ── 지도: 대중교통 공백 지수 (geopandas PNG — 반출 가능)
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# 왼쪽: 올빼미버스 커버리지
gdf_result.plot(
    column='owl_coverage_ratio',
    ax=axes[0],
    cmap='RdYlGn',
    legend=True,
    legend_kwds={'label': 'Owl Bus Coverage Ratio', 'shrink': 0.7},
    missing_kwds={'color': 'lightgrey'}
)
# 올빼미버스 정류장 표시
gdf_owl.plot(ax=axes[0], color='blue', markersize=2, alpha=0.5, label='Owl Bus Stop')
axes[0].set_title('Owl Bus Coverage by Dong')
axes[0].set_axis_off()

# 오른쪽: 대중교통 공백 지수
gdf_result.plot(
    column='대중교통공백_지수',
    ax=axes[1],
    cmap='Reds',
    legend=True,
    legend_kwds={'label': 'Transport Gap Index (0~1)', 'shrink': 0.7},
)
axes[1].set_title('Public Transport Gap Index by Dong')
axes[1].set_axis_off()

plt.suptitle('Seoul Late-Night Public Transport Gap Analysis', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + '02_transport_gap_map.png', dpi=150, bbox_inches='tight')
plt.close()
print('지도 저장 완료 → output/02_transport_gap_map.png')


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


...

findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


findfont: Font family 'NanumGothic' not found.


지도 저장 완료 → output/02_transport_gap_map.png


In [11]:
# ── 결과 저장 (4단계에서 사용)
df_save = df_transport.copy()
df_save.to_csv(OUTPUT_DIR + 'stage02_transport_gap.csv', index=False, encoding='utf-8-sig')

print(f'2단계 결과 저장 완료 → output/stage02_transport_gap.csv')
print(f'총 {len(df_save)}개 행정동 대중교통 공백 지수 산출')
print('컬럼:', list(df_save.columns))
print('출처: 서울시 버스노선정보(공개데이터포털) / 서울시 버스정류소위치정보')


2단계 결과 저장 완료 → output/stage02_transport_gap.csv
총 426개 행정동 대중교통 공백 지수 산출
컬럼: ['행정동코드', '행정동명', 'owl_covered', 'owl_coverage_ratio', '대중교통공백_지수']
출처: 서울시 버스노선정보(공개데이터포털) / 서울시 버스정류소위치정보
